# Test del modelo CNN 3D

Este notebook carga el checkpoint entrenado, evalua el conjunto `test` definido en `data/splits.json` y permite revisar que muestras ha acertado o fallado el modelo.

La salida del modelo se interpreta como probabilidad de tumor: `sigmoid(logit)`. Por defecto, se predice tumor cuando `score >= 0.5`.

In [1]:
from pathlib import Path
import json
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader


def find_repo_root(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "configs").exists():
            return candidate
    raise RuntimeError("No se encontro la raiz del repositorio")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.dataset_3d import BrainMRI3DDataset, load_splits
from src.evaluation.evaluate_3d import (
    average_precision,
    binary_auc,
    build_model_from_checkpoint,
    metrics_at_threshold,
    positive_probability,
    read_npz_metadata,
    threshold_for_target_sensitivity,
    torch_load,
)
from src.training.train_3d import configure_gpu_performance, load_config, parse_shape

print("Repo:", REPO_ROOT)

Repo: /media/uaxlab/Samsung USB/brain-mri-triage


## 1. Configuracion

Puedes cambiar `CHECKPOINT_PATH`, `SPLIT` o `THRESHOLD` si quieres probar otro modelo, otro split o un umbral distinto.

In [2]:
CONFIG_PATH = REPO_ROOT / "configs" / "train_3d.yaml"
SPLIT = "test"  # opciones: train, val, test
THRESHOLD = 0.5
BATCH_SIZE = 1

# --- selección de checkpoint ---
CHECKPOINT_BASE = REPO_ROOT / "outputs" / "checkpoints"

# runs con timestamp (YYYYMMDD_HHMMSS/best.pt)
ts_runs = sorted(
    [d for d in CHECKPOINT_BASE.iterdir() if d.is_dir() and (d / "best.pt").exists()],
    key=lambda d: d.name,
    reverse=True,
)

# fallback: nombre antiguo fijo
legacy = CHECKPOINT_BASE / "cnn3d_best.pt"

if ts_runs:
    print("Runs disponibles:")
    for i, r in enumerate(ts_runs):
        marker = " <-- última (default)" if i == 0 else ""
        print(f"  [{i}] {r.name}{marker}")
    # cambia el índice aquí para cargar un run distinto
    RUN_INDEX = 0
    CHECKPOINT_PATH = ts_runs[RUN_INDEX] / "best.pt"
elif legacy.exists():
    print("No hay runs con timestamp, usando checkpoint legado.")
    CHECKPOINT_PATH = legacy
else:
    raise FileNotFoundError(f"No se encontró ningún checkpoint en {CHECKPOINT_BASE}")

config = load_config(CONFIG_PATH)
splits = load_splits(REPO_ROOT / "data" / "splits.json")
file_paths = splits[SPLIT]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
perf_cfg = configure_gpu_performance(config, device)
channels_last_3d = device.type == "cuda" and bool(perf_cfg.get("channels_last_3d", True))
use_amp = bool(config.get("training", {}).get("amp", True)) and device.type == "cuda"

print("\nCheckpoint:", CHECKPOINT_PATH)
print("Split:", SPLIT)
print("Muestras:", len(file_paths))
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"No existe el checkpoint: {CHECKPOINT_PATH}")

No hay runs con timestamp, usando checkpoint legado.

Checkpoint: /media/uaxlab/Samsung USB/brain-mri-triage/outputs/checkpoints/cnn3d_best.pt
Split: test
Muestras: 340
Device: cuda
GPU: AMD Radeon RX 6700 XT


## 2. Cargar modelo y dataset de test

In [3]:
crop_shape = parse_shape(config.get("data", {}).get("crop_shape", [128, 160, 128]))
test_dataset = BrainMRI3DDataset(
    file_paths,
    crop_shape=crop_shape,
    random_crop=False,
    augment=False,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

checkpoint = torch_load(CHECKPOINT_PATH, device)
model = build_model_from_checkpoint(checkpoint, config, device)
if channels_last_3d:
    model = model.to(memory_format=torch.channels_last_3d)
model.eval()

print("Modelo cargado:", CHECKPOINT_PATH)
print("Crop shape:", crop_shape)

Modelo cargado: /media/uaxlab/Samsung USB/brain-mri-triage/outputs/checkpoints/cnn3d_best.pt
Crop shape: (128, 160, 128)


## 3. Ejecutar predicciones

Esta celda recorre todo el split de test y genera una tabla con una fila por muestra.

In [4]:
rows = []

with torch.inference_mode():
    for batch_idx, (volumes, labels) in enumerate(test_loader):
        volumes = volumes.to(device, non_blocking=True)
        if channels_last_3d:
            volumes = volumes.contiguous(memory_format=torch.channels_last_3d)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            logits = model(volumes)
            probs = positive_probability(logits)

        start = batch_idx * BATCH_SIZE
        for offset, (label, score) in enumerate(zip(labels.tolist(), probs.detach().cpu().tolist())):
            path = Path(file_paths[start + offset])
            metadata = read_npz_metadata(path)
            pred = int(score >= THRESHOLD)
            true = int(label)
            rows.append(
                {
                    "path": path,
                    "file": path.name,
                    "dataset": metadata["dataset"],
                    "subject_id": metadata["subject_id"],
                    "label": true,
                    "label_text": "tumor" if true == 1 else "no tumor",
                    "score_tumor": float(score),
                    "prediction": pred,
                    "prediction_text": "tumor" if pred == 1 else "no tumor",
                    "correct": pred == true,
                }
            )

df = pd.DataFrame(rows)
df.head()

/home/uaxlab/miniconda3/envs/igsan/lib/python3.11/site-packages/torch/nn/modules/linear.py:125: UserWarning: Attempting to use hipBLASLt on an unsupported architecture! Overriding blas backend to hipblas (Triggered internally at ../aten/src/ATen/Context.cpp:296.)
  return F.linear(input, self.weight, self.bias)


,path,file,dataset,subject_id,label,label_text,score_tumor,prediction,prediction_text,correct
0,/media/uaxlab/Samsung USB/brain-mri-triage/dat...,BraTS2021_00088.npz,brats,BraTS2021_00088,1,tumor,0.656250,1,tumor,True
1,/media/uaxlab/Samsung USB/brain-mri-triage/dat...,BraTS2021_00066.npz,brats,BraTS2021_00066,1,tumor,0.745117,1,tumor,True
2,/media/uaxlab/Samsung USB/brain-mri-triage/dat...,BraTS2021_00626.npz,brats,BraTS2021_00626,1,tumor,0.778809,1,tumor,True
3,/media/uaxlab/Samsung USB/brain-mri-triage/dat...,IXI381-Guys-1024.npz,ixi,IXI381-Guys-1024,0,no tumor,0.740723,1,tumor,False
4,/media/uaxlab/Samsung USB/brain-mri-triage/dat...,BraTS2021_00501.npz,brats,BraTS2021_00501,1,tumor,0.741211,1,tumor,True


## 4. Resumen de resultados

In [5]:
y_true = df["label"].astype(int).tolist()
y_score = df["score_tumor"].astype(float).tolist()

metrics = metrics_at_threshold(y_true, y_score, THRESHOLD)
summary = {
    "split": SPLIT,
    "n": len(df),
    "positivos": int(df["label"].sum()),
    "negativos": int((df["label"] == 0).sum()),
    "auc": binary_auc(y_true, y_score),
    "pr_auc": average_precision(y_true, y_score),
    **metrics,
}

pd.Series(summary)

split                    test
n                         340
positivos                 175
negativos                 165
auc                  0.745074
pr_auc               0.841922
threshold                 0.5
tp                        159
fp                        165
tn                          0
fn                         16
accuracy             0.467647
sensitivity          0.908571
specificity               0.0
precision            0.490741
npv                       0.0
f1                   0.637275
balanced_accuracy    0.454286
dtype: object

In [6]:
print("Recuento por dataset")
display(pd.crosstab(df["dataset"], df["label_text"], margins=True))

print("Aciertos/fallos por dataset")
display(pd.crosstab(df["dataset"], df["correct"], margins=True))

print("Errores")
display(df.loc[~df["correct"], ["file", "dataset", "label_text", "prediction_text", "score_tumor"]].sort_values("score_tumor", ascending=False))

Recuento por dataset


label_text,no tumor,tumor,All
dataset,,,
brats,0,87,87
ixi,91,0,91
nki_rockland,74,0,74
upenn,0,88,88
All,165,175,340


Aciertos/fallos por dataset


correct,False,True,All
dataset,,,
brats,2,85,87
ixi,91,0,91
nki_rockland,74,0,74
upenn,14,74,88
All,181,159,340


Errores


,file,dataset,label_text,prediction_text,score_tumor
294,IXI016-Guys-0697.npz,ixi,no tumor,tumor,0.790527
260,IXI063-Guys-0742.npz,ixi,no tumor,tumor,0.777344
153,IXI614-HH-2735.npz,ixi,no tumor,tumor,0.767578
160,NKI-A00059733-FLU1.npz,nki_rockland,no tumor,tumor,0.758301
53,IXI121-Guys-0772.npz,ixi,no tumor,tumor,0.755859
...,...,...,...,...,...
18,UPENN-GBM-00064.npz,upenn,tumor,no tumor,0.377197
102,UPENN-GBM-00307.npz,upenn,tumor,no tumor,0.347168
136,UPENN-GBM-00249.npz,upenn,tumor,no tumor,0.285645
309,UPENN-GBM-00621.npz,upenn,tumor,no tumor,0.251953


## 5. Revisar aciertos y errores

Cambia `VIEW_MODE` para visualizar ejemplos:

- `errors`: solo errores.
- `correct`: solo aciertos.
- `all`: cualquier muestra del test.

In [7]:
VIEW_MODE = "errors"  # errors, correct, all
N_EXAMPLES = 6
AXIS = 2


def best_slice_index(volume, axis=2, eps=1e-6):
    foreground = np.abs(volume) > eps
    if axis == 0:
        counts = foreground.sum(axis=(1, 2))
    elif axis == 1:
        counts = foreground.sum(axis=(0, 2))
    elif axis == 2:
        counts = foreground.sum(axis=(0, 1))
    else:
        raise ValueError("axis debe ser 0, 1 o 2")
    return int(np.argmax(counts)) if counts.max() else volume.shape[axis] // 2


def get_slice(volume, axis=2, index=None):
    if index is None:
        index = best_slice_index(volume, axis=axis)
    if axis == 0:
        return volume[index, :, :], index
    if axis == 1:
        return volume[:, index, :], index
    if axis == 2:
        return volume[:, :, index], index
    raise ValueError("axis debe ser 0, 1 o 2")


def robust_window(img, low=1, high=99, eps=1e-6):
    values = img[np.isfinite(img)]
    foreground = values[np.abs(values) > eps]
    values = foreground if foreground.size else values
    if values.size == 0:
        return 0.0, 1.0
    lo, hi = np.percentile(values, [low, high])
    if np.isclose(lo, hi):
        hi = lo + 1.0
    return float(lo), float(hi)


display_cmap = plt.cm.gray.copy()
display_cmap.set_bad(color="black")


def masked_for_display(img, eps=1e-6):
    rotated = np.rot90(img)
    return np.ma.masked_where(np.abs(rotated) <= eps, rotated)


def show_examples(rows_df, n_examples=6, axis=2):
    if rows_df.empty:
        print("No hay muestras para visualizar con este filtro")
        return

    sample_df = rows_df.sample(n=min(n_examples, len(rows_df)), random_state=42)
    n = len(sample_df)
    fig, axes = plt.subplots(n, 2, figsize=(9, max(3, 2.8 * n)))
    if n == 1:
        axes = np.array([axes])

    for row_idx, (_, row) in enumerate(sample_df.iterrows()):
        with np.load(row["path"]) as sample:
            t1 = sample["t1"]
            t2 = sample["t2"]

        combined = np.maximum(np.abs(t1), np.abs(t2))
        used_index = best_slice_index(combined, axis=axis)
        t1_slice, _ = get_slice(t1, axis=axis, index=used_index)
        t2_slice, _ = get_slice(t2, axis=axis, index=used_index)

        ok = "OK" if row["correct"] else "ERROR"
        title = (
            f"{ok} | real={row['label_text']} | pred={row['prediction_text']} | "
            f"score={row['score_tumor']:.3f} | {row['file']}"
        )

        for col_idx, (img, modality) in enumerate([(t1_slice, "T1"), (t2_slice, "T2")]):
            ax = axes[row_idx, col_idx]
            vmin, vmax = robust_window(img)
            ax.imshow(masked_for_display(img), cmap=display_cmap, vmin=vmin, vmax=vmax)
            ax.set_title(f"{modality} | slice={used_index}\n{title}", fontsize=8)
            ax.axis("off")

    plt.tight_layout()
    plt.show()


if VIEW_MODE == "errors":
    view_df = df.loc[~df["correct"]]
elif VIEW_MODE == "correct":
    view_df = df.loc[df["correct"]]
elif VIEW_MODE == "all":
    view_df = df
else:
    raise ValueError("VIEW_MODE debe ser: errors, correct o all")

show_examples(view_df, n_examples=N_EXAMPLES, axis=AXIS)

## 6. Guardar predicciones del notebook

In [8]:
OUTPUT_DIR = REPO_ROOT / "outputs" / "evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / f"cnn3d_{SPLIT}_notebook_predictions.csv"
json_path = OUTPUT_DIR / f"cnn3d_{SPLIT}_notebook_summary.json"

df_to_save = df.copy()
df_to_save["path"] = df_to_save["path"].astype(str)
df_to_save.to_csv(csv_path, index=False)
json_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("CSV guardado en:", csv_path)
print("Resumen guardado en:", json_path)

CSV guardado en: /media/uaxlab/Samsung USB/brain-mri-triage/outputs/evaluation/cnn3d_test_notebook_predictions.csv
Resumen guardado en: /media/uaxlab/Samsung USB/brain-mri-triage/outputs/evaluation/cnn3d_test_notebook_summary.json
